# Phase B: FinBERT Severity Enhancement

**목적**: V2Tone 기반 severity를 FinBERT 금융 감성 분석으로 정밀화

**필요 환경**: Colab GPU (T4 이상)

**입력**: `risk_events.parquet` (399만건)
**출력**: `risk_events_finbert.parquet` (severity_finbert 컬럼 추가)

---

## 0. 환경 설정

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 프로젝트 경로 설정 (본인 Drive 경로로 수정)
import os
PROJECT_DIR = '/content/drive/MyDrive/nabi_hyoghaw'  # ← 본인 경로로 수정
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# 의존성 설치
!pip install -q transformers torch pandas pyarrow tqdm

# GPU 확인
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## 1. FinBERT 모델 로드

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = 'ProsusAI/finbert'

print('Loading FinBERT...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

# GPU로 이동
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'FinBERT loaded on {device}')

## 2. risk_events 로드

In [ ]:
import pandas as pd
import numpy as np

# risk_events 로드 (title 컬럼 사용)
RISK_EVENTS_PATH = 'data/processed/risk_events.parquet'
df = pd.read_parquet(RISK_EVENTS_PATH, columns=['event_id', 'title', 'severity', 'risk_types'])
print(f'Loaded: {len(df):,} events')
print(f'Columns: {df.columns.tolist()}')

# title이 있는 행만 (없으면 URL이나 빈 문자열)
df['title'] = df['title'].fillna('').astype(str)
has_title = df['title'].str.len() > 10
print(f'With title (>10 chars): {has_title.sum():,} ({100*has_title.mean():.1f}%)')

## 3. 배치 FinBERT 추론

399만건을 GPU 배치로 처리합니다. T4 기준 약 30~60분 소요.

In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 64  # T4: 64, A100: 256

def finbert_batch_score(texts, tokenizer, model, device, batch_size=64):
    """
    FinBERT 배치 추론.
    Returns: (negative_scores, neutral_scores, positive_scores) as numpy arrays
    """
    n = len(texts)
    neg = np.zeros(n, dtype=np.float32)
    neu = np.zeros(n, dtype=np.float32)
    pos = np.zeros(n, dtype=np.float32)

    for i in tqdm(range(0, n, batch_size), desc='FinBERT'):
        batch = texts[i:i+batch_size]
        # 빈 문자열 처리
        batch = [t if len(t) > 5 else 'neutral news' for t in batch]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()

        neg[i:i+len(batch)] = probs[:, 0]  # negative
        neu[i:i+len(batch)] = probs[:, 1]  # neutral
        pos[i:i+len(batch)] = probs[:, 2]  # positive

    return neg, neu, pos

print(f'Processing {len(df):,} events with batch_size={BATCH_SIZE}...')
texts = df['title'].tolist()
neg, neu, pos = finbert_batch_score(texts, tokenizer, model, device, BATCH_SIZE)

df['finbert_negative'] = neg
df['finbert_neutral'] = neu
df['finbert_positive'] = pos
df['finbert_sentiment'] = pos - neg  # +1=positive, -1=negative

print(f'\nFinBERT 감성 분포:')
print(f'  Negative (< -0.3): {(df["finbert_sentiment"] < -0.3).sum():,}')
print(f'  Neutral  (-0.3~0.3): {(df["finbert_sentiment"].between(-0.3, 0.3)).sum():,}')
print(f'  Positive (> 0.3): {(df["finbert_sentiment"] > 0.3).sum():,}')

## 4. FinBERT 기반 Severity 재계산

In [ ]:
# severity 재계산: V2Tone severity + FinBERT 보정
# 부정적 뉴스 → severity 증가, 긍정적 뉴스 → severity 감소

FINBERT_WEIGHT = 0.3  # FinBERT 반영 비중

# FinBERT risk score: negative가 높을수록 리스크
finbert_risk = np.clip(neg - pos, -1.0, 1.0)  # [-1, 1]
finbert_adjustment = FINBERT_WEIGHT * finbert_risk * 2.0  # [-0.6, +0.6]

df['severity_finbert'] = np.clip(
    df['severity'] + finbert_adjustment,
    1.0, 5.0
).round(3)

# 비교
print('=== Severity 비교 ===')
print(f'  V2Tone severity mean: {df["severity"].mean():.3f}')
print(f'  FinBERT severity mean: {df["severity_finbert"].mean():.3f}')
print(f'  차이 (FinBERT - V2Tone): {(df["severity_finbert"] - df["severity"]).mean():.3f}')

# 예시 비교
print('\n=== 가장 큰 차이 (FinBERT가 severity를 올린 경우) ===')
df['_diff'] = df['severity_finbert'] - df['severity']
top_up = df.nlargest(5, '_diff')
for _, r in top_up.iterrows():
    print(f'  sev: {r["severity"]:.2f} → {r["severity_finbert"]:.2f} ({r["_diff"]:+.2f}) | {r["title"][:80]}')

print('\n=== 가장 큰 차이 (FinBERT가 severity를 내린 경우) ===')
top_down = df.nsmallest(5, '_diff')
for _, r in top_down.iterrows():
    print(f'  sev: {r["severity"]:.2f} → {r["severity_finbert"]:.2f} ({r["_diff"]:+.2f}) | {r["title"][:80]}')

## 5. 저장

In [ ]:
# 원본에 FinBERT 컬럼 추가해서 저장
full_df = pd.read_parquet(RISK_EVENTS_PATH)
full_df['finbert_negative'] = neg
full_df['finbert_neutral'] = neu
full_df['finbert_positive'] = pos
full_df['finbert_sentiment'] = pos - neg
full_df['severity_finbert'] = df['severity_finbert'].values

OUT_PATH = 'data/processed/risk_events_finbert.parquet'
full_df.to_parquet(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH} ({len(full_df):,} rows)')
print(f'New columns: finbert_negative, finbert_neutral, finbert_positive, finbert_sentiment, severity_finbert')

# 기존 파일 교체하려면:
# import shutil
# shutil.copy(OUT_PATH, RISK_EVENTS_PATH)
# print('Replaced original risk_events.parquet')

## 6. 검증: CAR에 미치는 영향 (선택)

In [ ]:
# FinBERT severity로 aggregate → CAR 비교
# (이 셀은 선택 사항 - 전체 파이프라인 재실행이 더 정확)

print('FinBERT 적용 후 파이프라인 재실행 방법:')
print('  1. risk_events_finbert.parquet → risk_events.parquet로 복사')
print('  2. severity 컬럼을 severity_finbert로 교체')
print('  3. python run_pipeline_remaining.py 실행')
print()
print('또는 base.yaml에서 use_finbert: true 설정 후 전체 재실행')